In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

# ==========================================
# 1. 读取数据与标签
# ==========================================
CSV_PATH = 'train_with_folds.csv'
df_train = pd.read_csv(CSV_PATH)
y_true = df_train['label_int'].values

# ==========================================
# 2. 读取各个模型的 OOF 预测概率 (训练集)
# ==========================================
oof_swin = np.load('models/oof_preds_swin.npy')
oof_effb3 = np.load('models/effb3/oof_preds_effb3.npy')

# ⏳ [三模型融合储备] 等 BEiT 训练完成后，取消下面这行的注释：
oof_beit = np.load('models/beit/oof_preds_beit.npy')

# ==========================================
# 3. 读取各个模型的测试集 Submission (推理集)
# ==========================================
cols = [f'c{i}' for i in range(10)]
sub_swin = pd.read_csv('models/submission_swin_5fold_fixed.csv')
sub_effb3 = pd.read_csv('models/effb3/submission_effb3_5fold.csv')

# ⏳ [三模型融合储备] 等 BEiT 推理完成后，取消下面这行的注释：
sub_beit = pd.read_csv('models/beit/submission_beit_5fold.csv')

# ==========================================
# 4. 将概率拼接成 Stacking 特征矩阵
# ==========================================
# 👉 当前：双模型拼接 (N x 20 维特征)
#X_train = np.hstack([oof_swin, oof_effb3])
#X_test = np.hstack([sub_swin[cols].values, sub_effb3[cols].values])

# ⏳ [三模型融合储备] 等 BEiT 就绪后，【注释掉上面两行】，并【取消下面两行的注释】 (变成 N x 30 维特征)
X_train = np.hstack([oof_swin, oof_effb3, oof_beit])
X_test = np.hstack([sub_swin[cols].values, sub_effb3[cols].values, sub_beit[cols].values])

# ==========================================
# 5. 使用 5 折交叉验证训练逻辑回归 (元模型)
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_meta = np.zeros((len(X_train), 10))
test_meta_preds = np.zeros((len(X_test), 10))

print(f"🧠 启动 Stacking 训练... 当前输入特征维度: {X_train.shape[1]}")
for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y_true)):
    X_trn, y_trn = X_train[trn_idx], y_true[trn_idx]
    X_val, y_val = X_train[val_idx], y_true[val_idx]
    
    # 逻辑回归元模型 (C=0.1 是极佳的正则化参数，防止过拟合)
    meta_model = LogisticRegression(max_iter=1000, C=0.1)
    meta_model.fit(X_trn, y_trn)
    
    # 预测当前折的验证集
    oof_meta[val_idx] = meta_model.predict_proba(X_val)
    # 预测真实测试集并做平均累加
    test_meta_preds += meta_model.predict_proba(X_test) / 5

# ==========================================
# 6. 评估与保存
# ==========================================
stacking_loss = log_loss(y_true, oof_meta)
print(f"🔥 Stacking 融合后的极致 CV Log Loss: {stacking_loss:.5f}")

# 借用第一个 submission 的外壳存放最终结果
final_sub = sub_swin.copy()
final_sub[cols] = test_meta_preds
final_sub.to_csv('models/stacking_ensemble_submission.csv', index=False)
print("🎉 Stacking 提交文件已生成！快去交一波看看成绩吧！")

🧠 启动 Stacking 训练... 当前输入特征维度: 30
🔥 Stacking 融合后的极致 CV Log Loss: 0.18917
🎉 Stacking 提交文件已生成！快去交一波看看成绩吧！
